# Ручная проверка ingestion-пайплайнов

Цель: пройти каждый ingestion от прямого запроса к API до результата, сохранённого
нашим `run(...)`. Ничего не меняет в репозитории — только диагностика.

Запускать из корня проекта (там, где `pyproject.toml`), например:

```
uv run jupyter lab
```

Ячейки идут по источникам: **EEA stations → EEA measurements → TED notices → TED codelists**.
Внутри каждого источника — 4 шага: (1) прямой запрос к API, (2) осмотр ответа,
(3) запуск нашего `run(...)`, (4) чтение сохранённого файла обратно.


In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import requests

# Проект должен быть в PYTHONPATH — если notebook лежит в notebooks/, поднимаемся на уровень выше
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "ingestion").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT)


---
# 1. EEA station metadata

Источник: ArcGIS FeatureServer. Один endpoint, без пагинации по датам — просто список станций.


## 1.1 Прямой запрос к API

- **Метод:** `GET`
- **URL:** `https://air.discomap.eea.europa.eu/arcgis/rest/services/AirQuality/AirQualityDownloadServiceEUMonitoringStations/MapServer/0/query`
- **Параметры:** `where` (серверный фильтр по стране), `outFields=*`, `returnGeometry=true`, `outSR=4326` (WGS84 lat/lon), `f=json`, `resultOffset`/`resultRecordCount` (пагинация)

Для ручной проверки уменьшаю `resultRecordCount` до 5, чтобы не тянуть все станции —
в реальном ingestion используется 2000 (см. `STATIONS_PAGE_SIZE` в `ingestion/eea/stations.py`).


In [ ]:
STATIONS_ENDPOINT = (
    "https://air.discomap.eea.europa.eu/arcgis/rest/services/AirQuality/"
    "AirQualityDownloadServiceEUMonitoringStations/MapServer/0/query"
)

params = {
    "where": "CountryCode='DE'",   # серверный фильтр — как в ingestion/eea/stations.py
    "outFields": "*",
    "returnGeometry": "true",
    "outSR": "4326",
    "f": "json",
    "resultOffset": 0,
    "resultRecordCount": 5,        # маленькая страница, только для ручной проверки
}

response = requests.get(STATIONS_ENDPOINT, params=params, timeout=30)
print("status:", response.status_code)
print("content-type:", response.headers.get("content-type"))


## 1.2 Осмотр ответа

Ответ — JSON. Нас интересует ключ `features` (список станций) и флаг `exceededTransferLimit`
(по нему ingestion понимает, есть ли ещё страницы). Каждая `feature` — это `attributes` +
`geometry` (наш код читает именно эти два блока в `fetch_raw_features()`).


In [ ]:
data = response.json()
print("top-level keys:", list(data.keys()))
print("exceededTransferLimit:", data.get("exceededTransferLimit"))
print("features count in this page:", len(data.get("features", [])))
print()
print("first feature (raw):")
print(json.dumps(data["features"][0], indent=2, ensure_ascii=False))


In [ ]:
# Какие именно поля attributes мы используем дальше (в normalization):
# CountryCode, AirQualityStationEoICode, PopupInfo — плюс geometry.x/y
first_attrs = data["features"][0]["attributes"]
print("attributes keys:", list(first_attrs.keys()))
print("geometry:", data["features"][0].get("geometry"))


## 1.3 Запуск нашего ingestion

Единственный режим — `mode="stations"`, без дат и без параметра страны (страна уже
зашита в код как `COUNTRY = "DE"`). Внутри пройдёт полная пагинация по 2000 записей за раз.


In [ ]:
from ingestion.eea.stations import run as run_eea_stations

run_eea_stations(mode="stations")


## 1.4 Проверка сохранённого файла

Результат — **сырой JSON** (список фич, как из API, без flatten) в
`data/raw/eea/stations/stations_raw.json`.


In [ ]:
stations_raw_path = PROJECT_ROOT / "data/raw/eea/stations/stations_raw.json"
print("exists:", stations_raw_path.exists())
print("size KB:", stations_raw_path.stat().st_size / 1024 if stations_raw_path.exists() else None)

features = json.loads(stations_raw_path.read_text())
print("features saved:", len(features))
print()
print("sample feature:")
print(json.dumps(features[0], indent=2, ensure_ascii=False))


In [ ]:
# Быстрая сводка без нормализации — просто чтобы глазами увидеть данные
sample_df = pd.DataFrame([f["attributes"] for f in features[:10]])
sample_df["longitude"] = [f["geometry"]["x"] for f in features[:10]]
sample_df["latitude"] = [f["geometry"]["y"] for f in features[:10]]
print("columns:", list(sample_df.columns))
sample_df[["AirQualityStationEoICode", "AQStationName", "CountryCode", "longitude", "latitude"]].head(10)


---
# 2. EEA measurements

Источник: Azure-хостед download API. Два шага: (1) POST — получить список URL на parquet-файлы,
(2) GET каждого URL — скачать сами данные.


## 2.1 Прямой запрос к API

**Шаг 1 — список файлов:**
- **Метод:** `POST`
- **URL:** `https://eeadmz1-downloads-api-appservice.azurewebsites.net/ParquetFile/urls`
- **Body:** `countries` (серверный фильтр), `pollutants`, `dataset`, `dateTimeStart`/`dateTimeEnd`, `aggregationType`, `source`

Беру маленькое окно (1 день) и один pollutant, чтобы не тянуть лишнее.


In [ ]:
URLS_ENDPOINT = "https://eeadmz1-downloads-api-appservice.azurewebsites.net/ParquetFile/urls"

payload = {
    "countries": ["DE"],           # серверный фильтр — как в ingestion/eea/measurements.py
    "cities": [],
    "pollutants": ["PM10"],        # один pollutant для маленького теста
    "dataset": 1,                  # E2a / Unverified / UTD
    "dateTimeStart": "2026-01-01",
    "dateTimeEnd": "2026-01-02",   # всего 1 день
    "aggregationType": "day",
    "source": "API",
}

response = requests.post(URLS_ENDPOINT, json=payload, timeout=30)
print("status:", response.status_code)
print("content-type:", response.headers.get("content-type"))


## 2.2 Осмотр ответа

Это **не сами данные**, а список URL на parquet-файлы (или иногда текст с одним URL на строку —
наш код `get_file_urls()` обрабатывает оба варианта, декодируя `utf-8-sig`).


In [ ]:
raw_text = response.content.decode("utf-8-sig", errors="replace")
try:
    urls_data = json.loads(raw_text)
except ValueError:
    urls_data = raw_text

print("response type:", type(urls_data))
if isinstance(urls_data, list):
    urls = urls_data
elif isinstance(urls_data, dict):
    urls = urls_data.get("urls", [])
else:
    urls = [l.strip() for l in urls_data.splitlines() if l.strip().lower().startswith("http")]

print("file count:", len(urls))
print("first few URLs:")
for u in urls[:3]:
    print(" ", u)


In [ ]:
# Скачиваем только первый файл, чтобы посмотреть на формат (не весь список)
if urls:
    file_response = requests.get(urls[0], timeout=60)
    print("status:", file_response.status_code)
    print("size KB:", len(file_response.content) / 1024)

    tmp_path = PROJECT_ROOT / "data/raw/eea/test/_manual_check.parquet"
    tmp_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path.write_bytes(file_response.content)

    preview_df = pd.read_parquet(tmp_path)
    print("columns:", list(preview_df.columns))
    preview_df.head(5)
else:
    print("Пустой список URL — проверьте дату (возможно, для 2026-01-01 данных ещё нет)")


## 2.3 Запуск нашего ingestion

Рекомендую сначала **`mode="test"`** — маленькое окно (последние 5 дней), один pollutant,
пишет в `data/raw/eea/test/`, не трогает основное хранилище.


In [ ]:
from ingestion.eea.measurements import run as run_eea_measurements

run_eea_measurements(mode="test")


Пример "Германия за 2026 год" — как просили в задаче. **Внимание:** это `historical` за
целый год, реально скачает много parquet-файлов (по одному запросу в API на весь год,
затем по одному GET на файл) и может занять заметное время. Если нужно быстрее —
замените `to_year` на `from_year` и сначала попробуйте с меньшим окном через `get_file_urls`
напрямую, как в шаге 2.1.


In [ ]:
run_eea_measurements(mode="historical", from_year=2026, to_year=2026)


## 2.4 Проверка сохранённых файлов

- `mode="test"` пишет один parquet в `data/raw/eea/test/`
- `mode="historical"` пишет по одному parquet-файлу на файл-от-API в `data/raw/eea/measurements/<year>/`
  плюс построчный `manifest.jsonl` с метаданными каждой закачки.


In [ ]:
manifest_path = PROJECT_ROOT / "data/raw/eea/measurements/manifest.jsonl"
print("manifest exists:", manifest_path.exists())

if manifest_path.exists():
    manifest_entries = [json.loads(line) for line in manifest_path.read_text().splitlines() if line.strip()]
    print("manifest entries:", len(manifest_entries))
    print("sample entry:")
    print(json.dumps(manifest_entries[0], indent=2, ensure_ascii=False))


In [ ]:
year_dir = PROJECT_ROOT / "data/raw/eea/measurements/2026"
parquet_files = sorted(year_dir.glob("*.parquet")) if year_dir.exists() else []
print("parquet files for 2026:", len(parquet_files))

if parquet_files:
    df = pd.read_parquet(parquet_files[0])
    print("file:", parquet_files[0].name)
    print("columns:", list(df.columns))
    print("rows:", len(df))
    df.head(5)


In [ ]:
# Отдельно — файл от mode="test"
test_files = sorted((PROJECT_ROOT / "data/raw/eea/test").glob("*.parquet"))
print("test files:", [f.name for f in test_files])
if test_files:
    test_df = pd.read_parquet(test_files[-1])
    print("columns:", list(test_df.columns))
    test_df.head(5)


---
# 3. TED notices

Источник: TED API v3, один POST endpoint с пагинацией через `iterationNextToken`.


## 3.1 Прямой запрос к API

- **Метод:** `POST`
- **URL:** `https://api.ted.europa.eu/v3/notices/search`
- **Body:** `query` (TED expert-query синтаксис — страна/тип/CPV/дата фильтруются здесь же,
  на стороне API), `fields`, `limit`, `paginationMode`

Маленький тестовый запрос — `limit=3`, `paginationMode=ITERATION` (тот же режим, что в
`historical`/`incremental`, чтобы заодно проверить пагинацию).


In [ ]:
TED_URL = "https://api.ted.europa.eu/v3/notices/search"

payload = {
    "query": (
        "buyer-country=DEU AND notice-type=can-standard AND "
        "(classification-cpv=90000000 OR classification-cpv=71313000) AND "
        "publication-date>=20260101 AND publication-date<=20260131 "
        "SORT BY publication-date DESC"
    ),
    "fields": ["publication-number", "notice-title", "buyer-name", "publication-date"],
    "limit": 3,
    "scope": "ALL",
    "paginationMode": "ITERATION",
    "onlyLatestVersions": True,
}

response = requests.post(TED_URL, json=payload, timeout=30)
print("status:", response.status_code)


## 3.2 Осмотр ответа

Подтверждено вживую (2026-08-19): ключи верхнего уровня — `notices`, `totalNoticeCount`,
`iterationNextToken`, `timedOut`. Наш код читает `notices` (с fallback на `results`, который
на практике не встречается) и `iterationNextToken` для следующей страницы.


In [ ]:
data = response.json()
print("top-level keys:", list(data.keys()))
print("totalNoticeCount:", data.get("totalNoticeCount"))
print("timedOut:", data.get("timedOut"))
print("has iterationNextToken:", bool(data.get("iterationNextToken")))
print()
print("first notice:")
print(json.dumps(data["notices"][0], indent=2, ensure_ascii=False))


## 3.3 Запуск нашего ingestion

Сначала **`mode="test"`** — 1 запись, `PAGE_NUMBER`-пагинация, не трогает `state.json`/`notices.jsonl`.


In [ ]:
from ingestion.ted.notices import run as run_ted_notices

run_ted_notices(mode="test")


Пример "Германия за 2026 год" — `historical` с полным диапазоном дат. **Внимание:** это
полная пагинация по 250 записей за раз на весь год, может занять время в зависимости от
объёма извещений. Страна (`DEU`) уже зашита в `build_query()`, отдельно передавать не нужно.


In [ ]:
run_ted_notices(mode="historical", from_date="2026-01-01", to_date="2026-12-31")


## 3.4 Проверка сохранённых файлов

- `mode="test"` → `data/raw/ted/test_ingestion.json` (обычный JSON, список из 1 записи)
- `mode="historical"`/`incremental"` → `data/raw/ted/notices.jsonl` (по одной записи на строку)
  и `data/raw/ted/state.json` (курсор `last_successful_run_date`)


In [ ]:
test_path = PROJECT_ROOT / "data/raw/ted/test_ingestion.json"
print("exists:", test_path.exists())
if test_path.exists():
    test_notices = json.loads(test_path.read_text())
    print("notices in file:", len(test_notices))
    print(json.dumps(test_notices[0], indent=2, ensure_ascii=False))


In [ ]:
notices_path = PROJECT_ROOT / "data/raw/ted/notices.jsonl"
print("exists:", notices_path.exists())

if notices_path.exists():
    lines = notices_path.read_text().splitlines()
    print("total lines (notices):", len(lines))
    first_notice = json.loads(lines[0])
    print("keys in one notice:", list(first_notice.keys()))
    print()
    print(json.dumps(first_notice, indent=2, ensure_ascii=False))


In [ ]:
state_path = PROJECT_ROOT / "data/raw/ted/state.json"
print("exists:", state_path.exists())
if state_path.exists():
    print(json.loads(state_path.read_text()))


---
# 4. TED reference codelists

Источник: GitHub (`OP-TED/eForms-SDK`), статичные XML-файлы. **Не привязано к стране и к
2026 году** — это общеевропейские справочники кодов (страна/валюта/CPV/NUTS и т.д.), не
процессные факты. Поэтому здесь нет параметров даты/страны — `run()` без аргументов.


## 4.1 Прямой запрос к API

- **Метод:** `GET`
- **URL:** `https://raw.githubusercontent.com/OP-TED/eForms-SDK/main/codelists/<filename>.gc`

Беру один маленький справочник (`currency.gc`), а не весь список — самый большой из семи
(`cpv.gc`) содержит ~9000 кодов, не нужен для проверки формата.


In [ ]:
codelist_url = "https://raw.githubusercontent.com/OP-TED/eForms-SDK/main/codelists/currency.gc"

response = requests.get(codelist_url, timeout=30)
print("status:", response.status_code)
print("content-type:", response.headers.get("content-type"))
print("size bytes:", len(response.content))


## 4.2 Осмотр ответа

Это Genericode XML. Нас интересуют `ColumnSet/Column` (названия колонок) и
`SimpleCodeList/Row/Value/SimpleValue` (сами данные) — именно их читает
`parse_genericode()` в `normalization/ted/codelists.py`.


In [ ]:
print(response.text[:1000])


In [ ]:
import xml.etree.ElementTree as ET

root = ET.fromstring(response.content)
columns = [col.attrib["Id"] for col in root.findall(".//ColumnSet/Column")]
rows = root.findall(".//SimpleCodeList/Row")

print("root tag:", root.tag)
print("columns:", columns)
print("row count:", len(rows))

# первая строка вручную, как это делает parse_genericode()
first_row = rows[0]
values = [v.find("SimpleValue").text for v in first_row.findall("Value")]
print("first row values:", values)


## 4.3 Запуск нашего ingestion

`run()` без аргументов — качает все 7 справочников из `CODELISTS`. `cpv` (~9000 кодов) может
идти дольше остальных. Помните: `nuts.gc` — предположительное имя файла, не подтверждённое,
если он 404-ится, это ожидаемо (см. комментарий в `ingestion/ted/codelists.py`).


In [ ]:
from ingestion.ted.codelists import run as run_ted_codelists

run_ted_codelists()


## 4.4 Проверка сохранённых файлов

Результат — сырые XML-файлы (без парсинга) в `data/reference/ted/codelists/<id>.gc.xml`.


In [ ]:
codelists_dir = PROJECT_ROOT / "data/reference/ted/codelists"
xml_files = sorted(codelists_dir.glob("*.gc.xml")) if codelists_dir.exists() else []
print("saved codelist files:", [f.name for f in xml_files])


In [ ]:
# Читаем один файл обратно и парсим — чтобы убедиться, что сохранённый XML валиден
currency_path = codelists_dir / "currency.gc.xml"
print("exists:", currency_path.exists())
print("size bytes:", currency_path.stat().st_size if currency_path.exists() else None)

if currency_path.exists():
    saved_root = ET.fromstring(currency_path.read_bytes())
    saved_rows = saved_root.findall(".//SimpleCodeList/Row")
    print("rows in saved file:", len(saved_rows))
